In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/assermosa/wanees-samples/my_samples/negative/Record (online-voice-recorder.com) (17).mp3
/kaggle/input/datasets/assermosa/wanees-samples/my_samples/negative/Record (online-voice-recorder.com) (15).mp3
/kaggle/input/datasets/assermosa/wanees-samples/my_samples/negative/Record (online-voice-recorder.com) (23).mp3
/kaggle/input/datasets/assermosa/wanees-samples/my_samples/negative/Record (online-voice-recorder.com) (20).mp3
/kaggle/input/datasets/assermosa/wanees-samples/my_samples/negative/Record (online-voice-recorder.com) (18).mp3
/kaggle/input/datasets/assermosa/wanees-samples/my_samples/negative/Record (online-voice-recorder.com) (26).mp3
/kaggle/input/datasets/assermosa/wanees-samples/my_samples/negative/Record (online-voice-recorder.com) (21).mp3
/kaggle/input/datasets/assermosa/wanees-samples/my_samples/negative/Record (online-voice-recorder.com) (19).mp3
/kaggle/input/datasets/assermosa/wanees-samples/my_samples/negative/Record (online-voice-recorder.com) (

In [2]:
%%bash
pip install -q openWakeWord
pip install -q onnx onnxruntime
pip install -q tensorflow==2.13.0
pip install -q librosa soundfile audiomentations
pip install -q tf2onnx
apt-get install -q -y ffmpeg sox
echo "✅ All dependencies installed"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.8/455.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 14.3 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  libopencore-amrnb0 libopencore-amrwb0 libsox-fmt-alsa libsox-fmt-base
  libsox3 libwavpack1
Suggested packages:
  libsox-fmt-all
The following NEW packages will be 

ERROR: Could not find a version that satisfies the requirement tensorflow==2.13.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0)
ERROR: No matching distribution found for tensorflow==2.13.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.23 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metad

In [34]:
import os
import glob

# ── Expected structure after uploading to Kaggle ──────────────
# /kaggle/input/wanees-samples/positive/  (15 files)
# /kaggle/input/wanees-samples/negative/  (15 files)

POSITIVE_DIR  = "/kaggle/input/datasets/assermosa/wanees-samples/my_samples/positive"
NEGATIVE_DIR  = "/kaggle/input/datasets/assermosa/wanees-samples/my_samples/negative"
WORK_DIR      = "/kaggle/working"
PROCESSED_DIR = f"{WORK_DIR}/processed"
MODEL_DIR     = f"{WORK_DIR}/models"


for d in [
    PROCESSED_DIR,
    f"{PROCESSED_DIR}/positive",
    f"{PROCESSED_DIR}/negative",
    f"{PROCESSED_DIR}/neg_augmented",
    f"{PROCESSED_DIR}/extra_negatives",
    f"{PROCESSED_DIR}/extra_neg_augmented",
    f"{PROCESSED_DIR}/tts_positive",
    f"{PROCESSED_DIR}/tts_pos_augmented",
    AUG_DIR,
    MODEL_DIR,
]:
    os.makedirs(d, exist_ok=True)

# ── Count uploaded samples ─────────────────────────────────────
pos_files = glob.glob(f"{POSITIVE_DIR}/**/*.*", recursive=True)
neg_files = glob.glob(f"{NEGATIVE_DIR}/**/*.*", recursive=True)

print(f"✅ Positive samples found : {len(pos_files)}")
print(f"✅ Negative samples found : {len(neg_files)}")
print()
for f in pos_files:
    print(f"   [POS] {os.path.basename(f)}")
for f in neg_files:
    print(f"   [NEG] {os.path.basename(f)}")

✅ Positive samples found : 14
✅ Negative samples found : 13

   [POS] Record (online-voice-recorder.com) (12).mp3
   [POS] Record (online-voice-recorder.com) (1).mp3
   [POS] Record (online-voice-recorder.com) (10).mp3
   [POS] Record (online-voice-recorder.com) (13).mp3
   [POS] Record (online-voice-recorder.com) (7).mp3
   [POS] Record (online-voice-recorder.com) (4).mp3
   [POS] Record (online-voice-recorder.com) (8).mp3
   [POS] Record (online-voice-recorder.com) (14).mp3
   [POS] Record (online-voice-recorder.com) (3).mp3
   [POS] Record (online-voice-recorder.com) (2).mp3
   [POS] Record (online-voice-recorder.com) (11).mp3
   [POS] Record (online-voice-recorder.com) (9).mp3
   [POS] Record (online-voice-recorder.com) (6).mp3
   [POS] Record (online-voice-recorder.com) (5).mp3
   [NEG] Record (online-voice-recorder.com) (17).mp3
   [NEG] Record (online-voice-recorder.com) (15).mp3
   [NEG] Record (online-voice-recorder.com) (23).mp3
   [NEG] Record (online-voice-recorder.com) (20

In [36]:
import librosa
import soundfile as sf
import numpy as np
import os
import glob

def preprocess_audio(input_path: str, output_path: str,
                     target_sr: int = 16000,
                     target_duration: float = 1.5) -> bool:
    try:
        audio, sr = librosa.load(input_path, sr=target_sr, mono=True)

        # Normalize
        if np.max(np.abs(audio)) > 0:
            audio = audio / np.max(np.abs(audio)) * 0.95

        # Pad or trim to fixed length
        target_samples = int(target_sr * target_duration)
        if len(audio) < target_samples:
            audio = np.pad(audio, (0, target_samples - len(audio)))
        else:
            audio = audio[:target_samples]

        # Save as 16-bit PCM WAV
        audio_int16 = (audio * 32767).astype(np.int16)
        sf.write(output_path, audio_int16, target_sr, subtype='PCM_16')
        return True

    except Exception as e:
        print(f"   ❌ Failed: {os.path.basename(input_path)} → {e}")
        return False


print("🔄 Preprocessing uploaded samples...\n")

for label, src_dir, dst_dir in [
    ("positive", POSITIVE_DIR, f"{PROCESSED_DIR}/positive"),
    ("negative", NEGATIVE_DIR, f"{PROCESSED_DIR}/negative"),
]:
    files   = glob.glob(f"{src_dir}/**/*.*", recursive=True)
    files   = [f for f in files if not f.endswith('.txt')]
    success = 0

    for i, fpath in enumerate(files):
        out_path = os.path.join(dst_dir, f"{label}_{i+1:03d}.wav")
        if preprocess_audio(fpath, out_path):
            success += 1
            data, sr = sf.read(out_path)
            print(f"   ✅ [{label}] {os.path.basename(fpath)} "
                  f"→ {os.path.basename(out_path)} "
                  f"({len(data)/sr:.2f}s)")

    print(f"\n✅ {label}: {success}/{len(files)} processed\n")

🔄 Preprocessing uploaded samples...

   ✅ [positive] Record (online-voice-recorder.com) (12).mp3 → positive_001.wav (1.50s)
   ✅ [positive] Record (online-voice-recorder.com) (1).mp3 → positive_002.wav (1.50s)
   ✅ [positive] Record (online-voice-recorder.com) (10).mp3 → positive_003.wav (1.50s)
   ✅ [positive] Record (online-voice-recorder.com) (13).mp3 → positive_004.wav (1.50s)
   ✅ [positive] Record (online-voice-recorder.com) (7).mp3 → positive_005.wav (1.50s)
   ✅ [positive] Record (online-voice-recorder.com) (4).mp3 → positive_006.wav (1.50s)
   ✅ [positive] Record (online-voice-recorder.com) (8).mp3 → positive_007.wav (1.50s)
   ✅ [positive] Record (online-voice-recorder.com) (14).mp3 → positive_008.wav (1.50s)
   ✅ [positive] Record (online-voice-recorder.com) (3).mp3 → positive_009.wav (1.50s)
   ✅ [positive] Record (online-voice-recorder.com) (2).mp3 → positive_010.wav (1.50s)
   ✅ [positive] Record (online-voice-recorder.com) (11).mp3 → positive_011.wav (1.50s)
   ✅ [positi

In [37]:
import gtts
import os

print("🎤 Generating synthetic Arabic NEGATIVE samples via TTS...\n")

NEGATIVE_PHRASES = [
    "صباح الفل", "السلام عليكم", "إزيك", "صباح الخير",
    "أنا تمام شكراً", "اسمك إيه", "ساكن فين",
    "الجو حلو النهاردة", "بحب القهوة", "تسلم كتير",
    "لو سمحت", "أيوه", "لأ", "يمكن", "أحمد", "محمد",
    "علي", "سارة", "فاطمة", "عمر", "خالد", "ياسمين",
    "الله أكبر", "بسم الله", "إن شاء الله", "ماشي",
    "تمام", "حلو", "قشطة", "يلا بينا",
]
extra_dir = f"{PROCESSED_DIR}/extra_negatives"
success   = 0

for i, phrase in enumerate(NEGATIVE_PHRASES):
    mp3_path = f"{extra_dir}/tts_{i:03d}.mp3"
    wav_path = f"{extra_dir}/tts_{i:03d}.wav"
    try:
        tts = gtts.gTTS(text=phrase, lang='ar', slow=False)
        tts.save(mp3_path)
        if preprocess_audio(mp3_path, wav_path):
            os.remove(mp3_path)
            success += 1
            print(f"   ✅ {phrase}")
    except Exception as e:
        print(f"   ❌ {phrase} → {e}")

print(f"\n✅ Generated {success}/{len(NEGATIVE_PHRASES)} TTS negative samples")

🎤 Generating synthetic Arabic NEGATIVE samples via TTS...

   ✅ صباح الفل
   ✅ السلام عليكم
   ✅ إزيك
   ✅ صباح الخير
   ✅ أنا تمام شكراً
   ✅ اسمك إيه
   ✅ ساكن فين
   ✅ الجو حلو النهاردة
   ✅ بحب القهوة
   ✅ تسلم كتير
   ✅ لو سمحت
   ✅ أيوه
   ✅ لأ
   ✅ يمكن
   ✅ أحمد
   ✅ محمد
   ✅ علي
   ✅ سارة
   ✅ فاطمة
   ✅ عمر
   ✅ خالد
   ✅ ياسمين
   ✅ الله أكبر
   ✅ بسم الله
   ✅ إن شاء الله
   ✅ ماشي
   ✅ تمام
   ✅ حلو
   ✅ قشطة
   ✅ يلا بينا

✅ Generated 30/30 TTS negative samples


In [38]:
import gtts
import os

print("🎤 Generating synthetic POSITIVE samples (ونيس) via TTS...\n")

POSITIVE_PHRASES = [
    "ونيس",
    "يا ونيس",
    "ونيس يا ونيس",
    "إيه رأيك يا ونيس؟",
    "كلمني يا ونيس",
    "يا ونيس أنا تعبان شوية",
    "ساعدني يا ونيس",
    "يا ونيس عايز أفضفض معاك",
    "إنت هنا يا ونيس؟",
    "يا ونيس سامعني؟",
    "أنا جيت يا ونيس",
    "صباح الفل يا ونيس",
    "يا ونيس محتاجك في حاجة",
    "ونيس يا غالي",
    "إلحقني يا ونيس",
    "يا ونيس قولي أعمل إيه",
    "ونيس حبيب قلبي",
    "يا ونيس رُد عليا",
    "إيه الأخبار يا ونيس؟",
    "ونيس يا صاحبي"
]

tts_pos_dir = f"{PROCESSED_DIR}/tts_positive"
success     = 0

# Normal speed
for i, phrase in enumerate(POSITIVE_PHRASES):
    mp3_path = f"{tts_pos_dir}/tts_pos_{i:03d}.mp3"
    wav_path = f"{tts_pos_dir}/tts_pos_{i:03d}.wav"
    try:
        tts = gtts.gTTS(text=phrase, lang='ar', slow=False)
        tts.save(mp3_path)
        if preprocess_audio(mp3_path, wav_path):
            os.remove(mp3_path)
            success += 1
            print(f"   ✅ [normal] {phrase}")
    except Exception as e:
        print(f"   ❌ {phrase} → {e}")

# Slow speed (different prosody = more variety)
for i, phrase in enumerate(POSITIVE_PHRASES):
    mp3_path = f"{tts_pos_dir}/tts_pos_slow_{i:03d}.mp3"
    wav_path = f"{tts_pos_dir}/tts_pos_slow_{i:03d}.wav"
    try:
        tts = gtts.gTTS(text=phrase, lang='ar', slow=True)
        tts.save(mp3_path)
        if preprocess_audio(mp3_path, wav_path):
            os.remove(mp3_path)
            success += 1
            print(f"   ✅ [slow]   {phrase}")
    except Exception as e:
        print(f"   ❌ [slow] {phrase} → {e}")

print(f"\n✅ Generated {success}/{len(POSITIVE_PHRASES) * 2} positive TTS samples")

🎤 Generating synthetic POSITIVE samples (ونيس) via TTS...

   ✅ [normal] ونيس
   ✅ [normal] يا ونيس
   ✅ [normal] ونيس يا ونيس
   ✅ [normal] إيه رأيك يا ونيس؟
   ✅ [normal] كلمني يا ونيس
   ✅ [normal] يا ونيس أنا تعبان شوية
   ✅ [normal] ساعدني يا ونيس
   ✅ [normal] يا ونيس عايز أفضفض معاك
   ✅ [normal] إنت هنا يا ونيس؟
   ✅ [normal] يا ونيس سامعني؟
   ✅ [normal] أنا جيت يا ونيس
   ✅ [normal] صباح الفل يا ونيس
   ✅ [normal] يا ونيس محتاجك في حاجة
   ✅ [normal] ونيس يا غالي
   ✅ [normal] إلحقني يا ونيس
   ✅ [normal] يا ونيس قولي أعمل إيه
   ✅ [normal] ونيس حبيب قلبي
   ✅ [normal] يا ونيس رُد عليا
   ✅ [normal] إيه الأخبار يا ونيس؟
   ✅ [normal] ونيس يا صاحبي
   ✅ [slow]   ونيس
   ✅ [slow]   يا ونيس
   ✅ [slow]   ونيس يا ونيس
   ✅ [slow]   إيه رأيك يا ونيس؟
   ✅ [slow]   كلمني يا ونيس
   ✅ [slow]   يا ونيس أنا تعبان شوية
   ✅ [slow]   ساعدني يا ونيس
   ✅ [slow]   يا ونيس عايز أفضفض معاك
   ✅ [slow]   إنت هنا يا ونيس؟
   ✅ [slow]   يا ونيس سامعني؟
   ✅ [slow]   أنا جيت يا ونيس
   ✅ [slow]

In [39]:
import numpy as np
import soundfile as sf
import os
import glob


def augment_audio(audio: np.ndarray, sr: int = 16000) -> list:
    augmented = []
    audio_f   = audio.astype(np.float32) / 32768.0

    # 1. Original
    augmented.append(audio_f.copy())
    # 2. Light noise
    augmented.append(np.clip(
        audio_f + np.random.normal(0, 0.005, len(audio_f)), -1, 1))
    # 3. Heavy noise
    augmented.append(np.clip(
        audio_f + np.random.normal(0, 0.015, len(audio_f)), -1, 1))
    # 4. Shift left
    augmented.append(np.roll(audio_f, -int(0.1 * sr)))
    # 5. Shift right
    augmented.append(np.roll(audio_f,  int(0.1 * sr)))
    # 6. Volume up
    augmented.append(np.clip(audio_f * 1.3, -1, 1))
    # 7. Volume down
    augmented.append(audio_f * 0.6)
    # 8. Slight speed up
    stretched = np.interp(
        np.linspace(0, len(audio_f), int(len(audio_f) * 0.95)),
        np.arange(len(audio_f)), audio_f)
    augmented.append(np.pad(stretched, (0, len(audio_f) - len(stretched))))
    # 9. Slight slow down
    augmented.append(
        np.interp(
            np.linspace(0, len(audio_f), int(len(audio_f) * 1.05)),
            np.arange(len(audio_f)), audio_f)[:len(audio_f)])
    # 10. Mild reverb
    impulse = np.exp(-np.linspace(0, 5, sr // 4))
    reverb  = np.convolve(audio_f, impulse / impulse.sum(),
                          mode='full')[:len(audio_f)]
    augmented.append(np.clip(reverb, -1, 1))

    return augmented  # 10 versions per file


def augment_folder(src_glob: str, dst_dir: str, label: str) -> int:
    os.makedirs(dst_dir, exist_ok=True)
    files = sorted(glob.glob(src_glob))
    count = 0
    for fpath in files:
        audio, sr = sf.read(fpath)
        if audio.dtype != np.int16:
            audio = (audio * 32767).astype(np.int16)
        base = os.path.splitext(os.path.basename(fpath))[0]
        for j, aug in enumerate(augment_audio(audio, sr)):
            out = f"{dst_dir}/{base}_aug{j:02d}.wav"
            sf.write(out,
                     (np.clip(aug, -1, 1) * 32767).astype(np.int16),
                     sr, subtype='PCM_16')
            count += 1
    print(f"   {label:35s}: {len(files):4d} files → {count:5d} augmented")
    return count


print("🔄 Augmenting all samples...\n")

pos_real_count = augment_folder(
    f"{PROCESSED_DIR}/positive/*.wav",
    AUG_DIR,
    "Positive real")

pos_tts_count = augment_folder(
    f"{PROCESSED_DIR}/tts_positive/*.wav",
    f"{PROCESSED_DIR}/tts_pos_augmented",
    "Positive TTS")

neg_real_count = augment_folder(
    f"{PROCESSED_DIR}/negative/*.wav",
    f"{PROCESSED_DIR}/neg_augmented",
    "Negative real")

neg_tts_count = augment_folder(
    f"{PROCESSED_DIR}/extra_negatives/*.wav",
    f"{PROCESSED_DIR}/extra_neg_augmented",
    "Negative TTS")

total_pos = pos_real_count + pos_tts_count
total_neg = neg_real_count + neg_tts_count

print(f"\n✅ Augmentation complete")
print(f"   Total positive : {total_pos}")
print(f"   Total negative : {total_neg}")
print(f"   Balance        : {total_pos/(total_pos+total_neg)*100:.1f}% positive")

if 40 <= total_pos/(total_pos+total_neg)*100 <= 60:
    print("   ✅ Balance is healthy")
else:
    print("   ⚠️  Imbalance detected — class weights will compensate")

🔄 Augmenting all samples...

   Positive real                      :   14 files →   140 augmented
   Positive TTS                       :   40 files →   400 augmented
   Negative real                      :   13 files →   130 augmented
   Negative TTS                       :   30 files →   300 augmented

✅ Augmentation complete
   Total positive : 540
   Total negative : 430
   Balance        : 55.7% positive
   ✅ Balance is healthy


In [40]:
import numpy as np
import soundfile as sf
import glob
import os
from openwakeword.utils import AudioFeatures

print("⏳ Loading AudioFeatures...")
audio_features = AudioFeatures()
print("✅ Loaded\n")


def extract_embedding(wav_path: str, af: AudioFeatures,
                      clip_length: int = 16000) -> np.ndarray:
    audio, sr = sf.read(wav_path)
    if audio.dtype != np.int16:
        audio = (np.clip(audio, -1, 1) * 32767).astype(np.int16)
    if len(audio) < clip_length:
        audio = np.pad(audio, (0, clip_length - len(audio)))
    n_clips    = len(audio) // clip_length
    clips      = audio[:n_clips * clip_length].reshape(n_clips, clip_length)
    embeddings = af.embed_clips(clips)            # (N, frames, 96)
    return embeddings.mean(axis=(0, 1)).astype(np.float32)


# ── Sanity check ───────────────────────────────────────────────
test_file = glob.glob(f"{PROCESSED_DIR}/positive/*.wav")[0]
test_emb  = extract_embedding(test_file, audio_features)
print(f"Sanity check → shape: {test_emb.shape}  "
      f"all_zero: {np.all(test_emb == 0)}\n")
assert not np.all(test_emb == 0), "❌ Embedding is all zeros — stop here"

# ── All sources ────────────────────────────────────────────────
sources = [
    (f"{AUG_DIR}/*.wav",                             1, "Positive real augmented"),
    (f"{PROCESSED_DIR}/tts_pos_augmented/*.wav",     1, "Positive TTS augmented"),
    (f"{PROCESSED_DIR}/tts_positive/*.wav",          1, "Positive TTS original"),
    (f"{PROCESSED_DIR}/negative/*.wav",              0, "Negative real original"),
    (f"{PROCESSED_DIR}/neg_augmented/*.wav",         0, "Negative real augmented"),
    (f"{PROCESSED_DIR}/extra_negatives/*.wav",       0, "Negative TTS original"),
    (f"{PROCESSED_DIR}/extra_neg_augmented/*.wav",   0, "Negative TTS augmented"),
]

print("🔄 Extracting embeddings...\n")
X, y = [], []

for pattern, label, name in sources:
    files = sorted(glob.glob(pattern))
    print(f"  {name:35s} : {len(files)} files")
    for fpath in files:
        X.append(extract_embedding(fpath, audio_features))
        y.append(label)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print(f"\n✅ Extraction complete")
print(f"   X shape  : {X.shape}")
print(f"   Positive : {int(sum(y == 1))}")
print(f"   Negative : {int(sum(y == 0))}")
print(f"   Balance  : {sum(y==1)/len(y)*100:.1f}% positive")

zeros = sum(1 for row in X if np.all(row == 0))
print(f"   Zero emb : {zeros}  {'✅' if zeros == 0 else '⚠️  check those files'}")

⏳ Loading AudioFeatures...


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


✅ Loaded

Sanity check → shape: (96,)  all_zero: False

🔄 Extracting embeddings...

  Positive real augmented             : 140 files
  Positive TTS augmented              : 400 files
  Positive TTS original               : 40 files
  Negative real original              : 13 files
  Negative real augmented             : 130 files
  Negative TTS original               : 30 files
  Negative TTS augmented              : 300 files

✅ Extraction complete
   X shape  : (1053, 96)
   Positive : 580
   Negative : 473
   Balance  : 55.1% positive
   Zero emb : 0  ✅


In [41]:
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# ── Split ──────────────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# ── Class weights ──────────────────────────────────────────────
cw  = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: cw[0], 1: cw[1]}

print(f"Train   : {len(X_train)} samples")
print(f"Val     : {len(X_val)} samples")
print(f"Balance : {sum(y_train==1)/len(y_train)*100:.1f}% positive")
print(f"Weights : neg={cw[0]:.3f}  pos={cw[1]:.3f}\n")


# ── Model ──────────────────────────────────────────────────────
def build_model(input_dim: int = 96) -> keras.Model:
    return keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dense(1,  activation='sigmoid'),
    ], name="wanees_classifier")


model = build_model(X.shape[1])
model.summary()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=20,
        restore_best_weights=True, mode='max'),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=10, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint(
        filepath=f"{MODEL_DIR}/best_wanees.keras",
        monitor='val_auc', save_best_only=True, mode='max'),
]

# ── Train ──────────────────────────────────────────────────────
print("\n🏋️ Training...\n")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=16,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1,
)

# ── Evaluate ───────────────────────────────────────────────────
y_pred = (model.predict(X_val, verbose=0) > 0.5).astype(int).flatten()
cm     = confusion_matrix(y_val, y_pred)

print(f"\n📊 Confusion Matrix (threshold=0.5):")
print(f"              Predicted")
print(f"              NEG    POS")
print(f"Actual NEG  [ {cm[0][0]:3d}    {cm[0][1]:3d} ]")
print(f"Actual POS  [ {cm[1][0]:3d}    {cm[1][1]:3d} ]")
print()
print(classification_report(
    y_val, y_pred,
    target_names=['negative', 'positive (ونيس)']))

Train   : 842 samples
Val     : 211 samples
Balance : 55.1% positive
Weights : neg=1.114  pos=0.907



Model: "wanees_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 64)             │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,217 (36.00 KB)

 Trainable params: 9,025 (35.25 KB)

 Non-trainable params: 192 (768.00 B)


🏋️ Training...

Epoch 1/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 8s 68ms/step - accuracy: 0.4615 - auc: 0.6160 - loss: 1.0179 - val_accuracy: 0.6540 - val_auc: 0.8832 - val_loss: 0.6059 - learning_rate: 0.0010
Epoch 2/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6590 - auc: 0.8318 - loss: 0.5991 - val_accuracy: 0.8673 - val_auc: 0.9374 - val_loss: 0.3833 - learning_rate: 0.0010
Epoch 3/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7953 - auc: 0.9131 - loss: 0.4225 - val_accuracy: 0.9100 - val_auc: 0.9660 - val_loss: 0.2835 - learning_rate: 0.0010
Epoch 4/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8531 - auc: 0.9414 - loss: 0.3384 - val_accuracy: 0.9289 - val_auc: 0.9802 - val_loss: 0.2245 - learning_rate: 0.0010
Epoch 5/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8744 - auc: 0.9464 - loss: 0.3110 - val_accuracy: 0.9242 - val_auc: 0.9853 - val_loss: 0.1991 - learning_rate: 0.0010
Epoch 6/200
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.89

In [43]:
import numpy as np
from sklearn.metrics import confusion_matrix

print("🔍 Finding best detection threshold...\n")

y_probs = model.predict(X_val, verbose=0).flatten()

thresholds = [0.40, 0.45, 0.50, 0.55, 0.60,
              0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

print(f"{'Thresh':>8} {'Precision':>10} {'Recall':>8} "
      f"{'F1':>8} {'FP':>6} {'FN':>6} {'Acc':>8}")
print("─" * 62)

best_thresh = 0.5
best_f1     = 0

for thresh in thresholds:
    y_pred = (y_probs >= thresh).astype(int)
    cm     = confusion_matrix(y_val, y_pred)
    tn, fp = cm[0][0], cm[0][1]
    fn, tp = cm[1][0], cm[1][1]

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0)
    acc       = (tp + tn) / len(y_val)

    marker = " ◄ best" if f1 > best_f1 else ""
    if f1 > best_f1:
        best_f1     = f1
        best_thresh = thresh

    print(f"{thresh:>8.2f} {precision:>10.3f} {recall:>8.3f} "
          f"{f1:>8.3f} {fp:>6} {fn:>6} {acc:>8.3f}{marker}")

print(f"\n✅ Best threshold : {best_thresh}")
print(f"   Best F1 score  : {best_f1:.3f}")
print(f"\n👉 Update pi_main.py:")
print(f"   WAKEWORD_THRESH = {best_thresh}")

🔍 Finding best detection threshold...

  Thresh  Precision   Recall       F1     FP     FN      Acc
──────────────────────────────────────────────────────────────
    0.40      1.000    0.983    0.991      0      2    0.991 ◄ best
    0.45      1.000    0.983    0.991      0      2    0.991
    0.50      1.000    0.983    0.991      0      2    0.991
    0.55      1.000    0.983    0.991      0      2    0.991
    0.60      1.000    0.983    0.991      0      2    0.991
    0.65      1.000    0.983    0.991      0      2    0.991
    0.70      1.000    0.966    0.982      0      4    0.981
    0.75      1.000    0.957    0.978      0      5    0.976
    0.80      1.000    0.948    0.973      0      6    0.972
    0.85      1.000    0.931    0.964      0      8    0.962
    0.90      1.000    0.931    0.964      0      8    0.962

✅ Best threshold : 0.4
   Best F1 score  : 0.991

👉 Update pi_main.py:
   WAKEWORD_THRESH = 0.4


In [44]:
import tensorflow as tf
import os

print("⏳ Converting to TFLite...\n")

best_model = keras.models.load_model(f"{MODEL_DIR}/best_wanees.keras")

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations               = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

tflite_path = f"{WORK_DIR}/wanees.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

size_kb = os.path.getsize(tflite_path) / 1024
print(f"✅ Saved : wanees.tflite ({size_kb:.1f} KB)")
print(f"\n👉 Download from the Output tab on the right")
print(f"   Then copy to Raspberry Pi:")
print(f"   scp wanees.tflite pi@<PI_IP>:/home/pi/younes/")

⏳ Converting to TFLite...

INFO:tensorflow:Assets written to: /tmp/tmp0o2xoywa/assets


INFO:tensorflow:Assets written to: /tmp/tmp0o2xoywa/assets


Saved artifact at '/tmp/tmp0o2xoywa'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96), dtype=tf.float32, name='input_layer_2')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134653537185872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134653537184336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134653537185296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134653537184528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134653537179152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134653537179536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134653537186448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134653537180688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134653537185680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134653537185488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134653537185104: Tenso

W0000 00:00:1772228171.248550      55 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1772228171.248589      55 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1772228171.257258      55 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
